# DifIISR — setup, stage 1

Run **Runtime → Run all** with a GPU runtime selected.

This notebook checks the GPU, clones the official source at a fixed commit, and creates an isolated Python 3.10 environment. It does **not** install model dependencies, download weights/data, or run inference yet.

Existing source files and environments are preserved; unexpected state stops execution. No credentials are needed because the upstream code is public.

The Colab notebook kernel stays unchanged. Later model commands must explicitly use `/content/difiisr-env/bin/python`. Files under `/content` are temporary: rerun setup after a runtime reset.

GitHub is the canonical notebook source. Opening it in Colab does not automatically save changes back to GitHub. Do not publish tokens or sensitive data in notebook outputs.

Reference: https://github.com/zirui0625/DifIISR


In [ ]:
import sys
import subprocess
from pathlib import Path

def run(args, cwd=None):
    result = subprocess.run(args, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    print(result.stdout, flush=True)
    result.check_returncode()
    return result.stdout.strip()

print("=== 1. Colab GPU check ===", flush=True)
import torch
print("Notebook Python:", sys.version)
print("Notebook PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab, then run this cell again.")
print("GPU:", torch.cuda.get_device_name(0))

print("=== 2. Official source ===", flush=True)
repo = Path("/content/DifIISR")
expected_commit = "09ca97ea48d481656dd8090e84099c963059ac41"
if not repo.exists():
    run(["git", "clone", "https://github.com/zirui0625/DifIISR.git", str(repo)])
    run(["git", "checkout", "--detach", expected_commit], cwd=repo)
else:
    if not (repo / ".git").exists():
        raise RuntimeError("Existing /content/DifIISR is not a Git repository; nothing was changed.")
    current = run(["git", "rev-parse", "HEAD"], cwd=repo)
    if current != expected_commit:
        raise RuntimeError("Existing checkout differs from the pinned version; nothing was overwritten.")
status = run(["git", "status", "--porcelain"], cwd=repo)
if status:
    raise RuntimeError("Existing checkout has local changes. Review them before proceeding.")
print("Source commit:", expected_commit)

print("=== 3. Isolated Python environment ===", flush=True)
run([sys.executable, "-m", "pip", "install", "uv"])
run([sys.executable, "-m", "uv", "--version"])
env_dir = Path("/content/difiisr-env")
env_python = env_dir / "bin" / "python"
if not env_dir.exists():
    run([sys.executable, "-m", "uv", "venv", "--python", "3.10", str(env_dir)])
if not env_python.exists():
    raise RuntimeError("Existing environment is incomplete; no files were removed.")
run([str(env_python), "-c",
     "import sys; print(sys.version); assert sys.version_info[:2] == (3, 10), 'Expected Python 3.10'"])

print("=== Official requirements (not installed yet) ===", flush=True)
print((repo / "requirements.txt").read_text())
print("SETUP_STAGE_1_OK")
print("Python 3.10 is ready. Model dependencies, weights, and data are NOT installed yet.")
print("The notebook kernel still uses Colab Python; model commands must use:", env_python)
